In [2]:
import json

with open("../data/restaurants.json", "r", encoding="utf-8") as f:
  data_json = json.load(f)

print(type(data_json))
print(data_json)

<class 'dict'>
{'ville': 'Catane', 'pays': 'Italie', 'releve': '2026-11-05', 'restaurants': [{'nom': 'Gelateria Zio Pietro dal 1964', 'adresse': 'Via Porta di Ferro, 47, 95131 Catania CT', 'type': 'glacier', 'notes': [4.6, 4.8, 4.7]}, {'nom': "A Casa d'Amici", 'adresse': 'Via Fischetti, 14, 95131 Catania CT', 'type': 'trattoria', 'notes': [4.4, 4.5]}, {'nom': 'La Bitta', 'adresse': 'Via Acquicella Porto, 95121 Catania CT', 'type': 'poisson', 'notes': [4.2, 4.0, 4.3, 4.1]}, {'nom': 'Steak House', 'adresse': 'Via Porta di Ferro, 8, 95100 Catania CT', 'type': 'grill', 'notes': [3.9, 4.1]}]}


In [6]:
print(df_restaurants.columns)

Index(['nom', 'adresse', 'type', 'ville', 'pays'], dtype='str')


In [7]:
print(data_json["restaurants"][0])

{'nom': 'Gelateria Zio Pietro dal 1964', 'adresse': 'Via Porta di Ferro, 47, 95131 Catania CT', 'type': 'glacier', 'notes': [4.6, 4.8, 4.7]}


In [8]:
import json
import time
import pandas as pd
import requests

# --- ÉTAPE 1 : Du JSON au DataFrame (4 lignes) ---
with open("../data/restaurants.json", "r", encoding="utf-8") as f:
  data_json = json.load(f)

# Création directe du DataFrame depuis la liste des restaurants
df_restaurants = pd.DataFrame(data_json["restaurants"])

# Ajout des informations de ville et de pays présentes à la racine du JSON
df_restaurants["ville"] = data_json["ville"]
df_restaurants["pays"] = data_json["pays"]

# Réorganisation des colonnes dans l'ordre demandé
df_restaurants = df_restaurants[
    ["nom", "adresse", "type", "ville", "pays", "notes"]
]

# --- ÉTAPE 2 : La note moyenne ---
# Calcul de la moyenne des notes et arrondi à deux décimales
df_restaurants["note_moyenne"] = df_restaurants["notes"].apply(
    lambda x: round(sum(x) / len(x), 2)
)

display(df_restaurants)

,nom,adresse,type,ville,pays,notes,note_moyenne
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,Catane,Italie,"[4.6, 4.8, 4.7]",4.70
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,Catane,Italie,"[4.4, 4.5]",4.45
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,Catane,Italie,"[4.2, 4.0, 4.3, 4.1]",4.15
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,Catane,Italie,"[3.9, 4.1]",4.00


In [9]:
def geocoder_nominatim(adresse):
  url = "https://nominatim.openstreetmap.org/search"
  headers = {"User-Agent": "FormationDataAnalyst-ProjetRestaurants/1.0"}
  params = {"q": adresse, "format": "json", "limit": 1}

  try:
    reponse = requests.get(url, params=params, headers=headers, timeout=5)
    time.sleep(1)  # Temporisation obligatoire d'1 seconde

    if reponse.status_code == 200:
      resultats = reponse.json()
      if resultats:
        lat = float(resultats[0]["lat"])
        lon = float(resultats[0]["lon"])
        return lat, lon
  except requests.exceptions.RequestException:
    pass

  return None, None


# Application du géocodage sur la colonne adresse
latitudes = []
longitudes = []

for adresse in df_restaurants["adresse"]:
  lat, lon = geocoder_nominatim(adresse)
  latitudes.append(lat)
  longitudes.append(lon)

df_restaurants["lat"] = latitudes
df_restaurants["lon"] = longitudes

display(df_restaurants)

,nom,adresse,type,ville,pays,notes,note_moyenne,lat,lon
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,Catane,Italie,"[4.6, 4.8, 4.7]",4.70,37.502093,15.093195
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,Catane,Italie,"[4.4, 4.5]",4.45,37.506171,15.094322
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,Catane,Italie,"[4.2, 4.0, 4.3, 4.1]",4.15,37.489440,15.079431
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,Catane,Italie,"[3.9, 4.1]",4.00,37.502933,15.093041


In [10]:
pip install folium

Note: you may need to restart the kernel to use updated packages.


In [11]:
import folium

# --- ÉTAPE 4 : La carte interactive ---
# Centrage sur les coordonnées du premier restaurant
lat_centre = df_restaurants.loc[0, "lat"]
lon_centre = df_restaurants.loc[0, "lon"]

# Zoom élevé (15 ou 16) car les restaurants de Catane sont très proches
carte_catane = folium.Map(location=[lat_centre, lon_centre], zoom_start=15)

# Ajout d'un marqueur pour chaque établissement
for index, row in df_restaurants.iterrows():
  # Texte de l'infobulle avec le nom et la note moyenne
  popup_html = f"<b>{row['nom']}</b><br>Note moyenne : {row['note_moyenne']}"

  folium.Marker(
      location=[row["lat"], row["lon"]],
      popup=folium.Popup(popup_html, max_width=300),
      tooltip=row["nom"],
      icon=folium.Icon(color="orange", icon="cutlery", prefix="fa"),
  ).add_to(carte_catane)

# Affichage de la carte
display(carte_catane)


# --- ÉTAPE 5 : Sauvegarder le résultat ---
# Écriture du DataFrame complet dans le fichier CSV demandé
df_restaurants.to_csv("../data/restaurants_geocodes.csv", index=False)
print("DataFrame exporté avec succès vers ../data/restaurants_geocodes.csv")

DataFrame exporté avec succès vers ../data/restaurants_geocodes.csv
